# 06E — PCA Bootstrap Stability

## Objective

This notebook evaluates the sampling stability of the PCA structure identified in
06C–06D.

The scientific question is:

> Are the retained PCA subspaces stable when the set of recordings is perturbed
> through recording-level bootstrap resampling?

Bootstrap resampling is performed at the recording level, preserving the primary
unit of analysis established in the preceding notebooks.

For each PCA representation, a PCA model is fitted to the full dataset to define
the reference subspace. PCA is then refitted to bootstrap samples of the same
recordings, and the bootstrap PCA subspace is compared with the reference
subspace using principal angles.

This analysis evaluates sampling stability separately from the preprocessing
sensitivity examined in 06D.

## Important methodological choices

- The recording is the primary sampling unit.
- Bootstrap samples contain 275 recordings sampled with replacement.
- The feature representations produced in 06B are used without refitting their
  preprocessing transformations within each bootstrap replicate.
- This isolates sampling stability of the PCA representation rather than mixing
  sampling variation with preprocessing variation.
- The retained dimensionality for each representation is determined by the
  permutation parallel analysis from 06C.
- Principal angles are used because individual PCA axes can change sign and can
  rotate within a multidimensional subspace.
- Metadata and ecological labels are not used to construct the PCA.
- Random seeds are fixed for reproducibility.

In [1]:
# ============================================================
# 06E.1 — Imports and paths
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles


# -----------------------------
# Reproducibility
# -----------------------------
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


# -----------------------------
# Bootstrap configuration
# -----------------------------
N_BOOTSTRAP = 500
N_RECORDINGS = 275


# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path.cwd()

PCA_REP_DIR = PROJECT_ROOT / "data" / "processed" / "pca_representations"
PCA_ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed" / "pca_analysis"

OUTPUT_DIR = PCA_ANALYSIS_DIR / "bootstrap_stability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 70)
print("06E — PCA BOOTSTRAP STABILITY")
print("=" * 70)

print(f"Project root:       {PROJECT_ROOT}")
print(f"PCA representations: {PCA_REP_DIR}")
print(f"PCA analysis:        {PCA_ANALYSIS_DIR}")
print(f"Output directory:    {OUTPUT_DIR}")
print(f"Bootstrap replicates: {N_BOOTSTRAP}")
print(f"Random seed:          {RANDOM_SEED}")

06E — PCA BOOTSTRAP STABILITY
Project root:       c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks
PCA representations: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks\data\processed\pca_representations
PCA analysis:        c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks\data\processed\pca_analysis
Output directory:    c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks\data\processed\pca_analysis\bootstrap_stability
Bootstrap replicates: 500
Random seed:          42


In [3]:
# ============================================================
# 06E.2 — Locate existing PCA artifacts
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd()

print("=" * 70)
print("LOCATING EXISTING PCA ARTIFACTS")
print("=" * 70)

# Start from the current notebook directory and search upward
search_roots = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent.parent,
]

seen = set()

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        # Only show files plausibly related to PCA representations/analysis
        path_lower = str(path).lower()

        if (
            "pca" in path_lower
            and (
                path.suffix.lower() in {".npy", ".npz", ".csv", ".json", ".parquet", ".pkl"}
            )
        ):
            resolved = str(path.resolve())

            if resolved not in seen:
                seen.add(resolved)
                print(path)

print("\n" + "=" * 70)
print(f"Candidate PCA artifact files found: {len(seen)}")
print("=" * 70)

LOCATING EXISTING PCA ARTIFACTS
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_components.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_explained_variance.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_explained_variance_ratio.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_feature_names.json
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_pa_null_95pct.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_pa_null_mean.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_pa_observed_eigenvalues.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\baseline_pa_retained.npy
c:\Users\Sa

In [6]:
# ============================================================
# 06E.2 — Load and validate existing PCA artifacts
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

# Current notebook:
# project/research/notebooks
#
# Project root:
# project/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

PCA_ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed" / "pca_analysis"

OUTPUT_DIR = PCA_ANALYSIS_DIR / "bootstrap_stability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Representations
# ------------------------------------------------------------

REPRESENTATIONS = [
    "baseline",
    "distribution_aware",
    "yeo_johnson",
    "block_balanced",
    "ndsi_derived_excluded",
    "ndsi_family_excluded",
]


# Retained dimensions established by 06C permutation
# parallel analysis.
#
# IMPORTANT:
# *_pa_retained.npy contains a PC-level retention mask,
# not a scalar count. Therefore we use these validated
# 06C counts directly.
RETAINED_PCS = {
    "baseline": 9,
    "distribution_aware": 9,
    "yeo_johnson": 9,
    "block_balanced": 7,
    "ndsi_derived_excluded": 9,
    "ndsi_family_excluded": 9,
}


N_RECORDINGS = 275


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

REFERENCE_PCA = {}
FEATURE_NAMES = {}


# ------------------------------------------------------------
# Load PCA artifacts
# ------------------------------------------------------------

print("=" * 70)
print("LOADING EXISTING PCA ARTIFACTS")
print("=" * 70)

print(f"\nProject root:")
print(PROJECT_ROOT)

print(f"\nPCA analysis directory:")
print(PCA_ANALYSIS_DIR)


for name in REPRESENTATIONS:

    components_path = (
        PCA_ANALYSIS_DIR / f"{name}_components.npy"
    )

    scores_path = (
        PCA_ANALYSIS_DIR / f"{name}_scores.npy"
    )

    eigenvalues_path = (
        PCA_ANALYSIS_DIR / f"{name}_explained_variance.npy"
    )

    feature_names_path = (
        PCA_ANALYSIS_DIR / f"{name}_feature_names.json"
    )

    retained_mask_path = (
        PCA_ANALYSIS_DIR / f"{name}_pa_retained.npy"
    )


    # --------------------------------------------------------
    # File existence checks
    # --------------------------------------------------------

    required_paths = [
        components_path,
        scores_path,
        eigenvalues_path,
        feature_names_path,
        retained_mask_path,
    ]

    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(
                f"\nMissing expected PCA artifact:\n{path}"
            )


    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    components = np.load(components_path)
    scores = np.load(scores_path)
    eigenvalues = np.load(eigenvalues_path)

    with open(feature_names_path, "r", encoding="utf-8") as f:
        feature_names = json.load(f)

    # 06C saved a PC-level retention mask.
    retained_mask = np.asarray(
        np.load(retained_mask_path)
    ).astype(bool)


    # --------------------------------------------------------
    # Validate retention mask
    # --------------------------------------------------------

    if retained_mask.ndim != 1:
        raise ValueError(
            f"{name}: expected 1-D retention mask, "
            f"found shape {retained_mask.shape}"
        )

    if retained_mask.shape[0] != components.shape[1]:
        raise ValueError(
            f"{name}: retention mask length "
            f"({retained_mask.shape[0]}) does not match "
            f"number of PCs ({components.shape[1]})"
        )

    expected_pcs = RETAINED_PCS[name]

    mask_count = int(retained_mask.sum())

    if mask_count != expected_pcs:
        raise ValueError(
            f"{name}: retention mask indicates {mask_count} PCs, "
            f"but validated 06C result indicates {expected_pcs}"
        )


    # --------------------------------------------------------
    # Validate dimensions
    # --------------------------------------------------------

    if components.shape[0] != len(feature_names):
        raise ValueError(
            f"{name}: components/features mismatch: "
            f"{components.shape[0]} vs {len(feature_names)}"
        )

    if scores.shape[0] != N_RECORDINGS:
        raise ValueError(
            f"{name}: expected {N_RECORDINGS} recordings, "
            f"found {scores.shape[0]}"
        )

    if scores.shape[1] != components.shape[1]:
        raise ValueError(
            f"{name}: scores/components dimension mismatch"
        )

    if len(eigenvalues) != components.shape[1]:
        raise ValueError(
            f"{name}: eigenvalue/component mismatch"
        )


    # --------------------------------------------------------
    # Validate numerical values
    # --------------------------------------------------------

    if not np.isfinite(components).all():
        raise ValueError(
            f"{name}: components contain non-finite values"
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            f"{name}: scores contain non-finite values"
        )

    if not np.isfinite(eigenvalues).all():
        raise ValueError(
            f"{name}: eigenvalues contain non-finite values"
        )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    REFERENCE_PCA[name] = {
        "components": components,
        "scores": scores,
        "eigenvalues": eigenvalues,
        "retained_pcs": expected_pcs,
        "retained_mask": retained_mask,
    }

    FEATURE_NAMES[name] = feature_names


    print(
        f"{name:28s} "
        f"features={len(feature_names):2d} | "
        f"components={components.shape} | "
        f"scores={scores.shape} | "
        f"retained PCs={expected_pcs}"
    )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("06E.2 VALIDATION PASSED")
print("=" * 70)

print(f"Representations loaded: {len(REFERENCE_PCA)}")
print(f"Recordings:              {N_RECORDINGS}")
print(f"Output directory:        {OUTPUT_DIR}")

LOADING EXISTING PCA ARTIFACTS

Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

PCA analysis directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis
baseline                     features=50 | components=(50, 50) | scores=(275, 50) | retained PCs=9
distribution_aware           features=50 | components=(50, 50) | scores=(275, 50) | retained PCs=9
yeo_johnson                  features=50 | components=(50, 50) | scores=(275, 50) | retained PCs=9
block_balanced               features=50 | components=(50, 50) | scores=(275, 50) | retained PCs=7
ndsi_derived_excluded        features=48 | components=(48, 48) | scores=(275, 48) | retained PCs=9
ndsi_family_excluded         features=46 | components=(46, 46) | scores=(275, 46) | retained PCs=9

06E.2 VALIDATION PASSED
Representations loaded: 6
Recordings:              275
Output directory:        c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\p

In [8]:
# ============================================================
# 06E.3 — Locate the exact 06B representation matrices
# ============================================================

from pathlib import Path

print("=" * 70)
print("SEARCHING FOR 06B PCA REPRESENTATION MATRICES")
print("=" * 70)

project_root = Path.cwd().parent.parent

candidate_dirs = [
    project_root / "data" / "processed",
    project_root / "research" / "data" / "processed",
]

seen = set()

for base_dir in candidate_dirs:

    if not base_dir.exists():
        continue

    print(f"\nSearching: {base_dir}")

    for path in base_dir.rglob("*"):

        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix not in {".npy", ".npz", ".csv", ".parquet", ".pkl"}:
            continue

        name_lower = path.name.lower()
        path_lower = str(path).lower()

        # Look specifically for likely representation artifacts,
        # not the PCA component/score outputs we already loaded.
        if (
            any(term in name_lower for term in [
                "baseline",
                "distribution",
                "yeo",
                "block",
                "ndsi"
            ])
            and not any(term in name_lower for term in [
                "component",
                "score",
                "explained",
                "singular",
                "pa_null",
                "pa_observed",
                "pa_retained"
            ])
        ):

            resolved = str(path.resolve())

            if resolved not in seen:
                seen.add(resolved)
                print(path)

print("\n" + "=" * 70)
print(f"Candidate representation files found: {len(seen)}")
print("=" * 70)

SEARCHING FOR 06B PCA REPRESENTATION MATRICES

Searching: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\baseline.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\block_balanced.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\distribution_aware.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\ndsi_derived_excluded.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\ndsi_family_excluded.npy
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations\yeo_johnson.npy

Candidate representation files found: 6


In [9]:
# ============================================================
# 06E.3 — Load 06B representations and test
#             principal-angle calculation
# ============================================================

from scipy.linalg import subspace_angles


# ------------------------------------------------------------
# Representation directory
# ------------------------------------------------------------

REPRESENTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pca_representations"
)


# ------------------------------------------------------------
# Load the exact 06B representation matrices
# ------------------------------------------------------------

X_REPRESENTATIONS = {}

print("=" * 70)
print("LOADING 06B REPRESENTATION MATRICES")
print("=" * 70)

for name in REPRESENTATIONS:

    path = REPRESENTATION_DIR / f"{name}.npy"

    if not path.exists():
        raise FileNotFoundError(
            f"Missing 06B representation:\n{path}"
        )

    X = np.load(path)

    expected_features = len(FEATURE_NAMES[name])

    # --------------------------------------------------------
    # Structural validation
    # --------------------------------------------------------

    if X.shape[0] != N_RECORDINGS:
        raise ValueError(
            f"{name}: expected {N_RECORDINGS} recordings, "
            f"found {X.shape[0]}"
        )

    if X.shape[1] != expected_features:
        raise ValueError(
            f"{name}: expected {expected_features} features, "
            f"found {X.shape[1]}"
        )

    if not np.isfinite(X).all():
        raise ValueError(
            f"{name}: representation contains non-finite values"
        )

    X_REPRESENTATIONS[name] = X

    print(
        f"{name:28s} "
        f"shape={X.shape} | "
        f"features={X.shape[1]}"
    )


# ------------------------------------------------------------
# Construct reference PCA subspaces
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSTRUCTING REFERENCE PCA SUBSPACES")
print("=" * 70)

REFERENCE_SUBSPACES = {}

for name in REPRESENTATIONS:

    components = REFERENCE_PCA[name]["components"]
    k = REFERENCE_PCA[name]["retained_pcs"]

    # sklearn PCA components:
    #   (n_components, n_features)
    #
    # Subspace basis:
    #   (n_features, k)
    reference_basis = components[:k, :].T

    # Validate orthonormality.
    gram = reference_basis.T @ reference_basis

    orthogonality_error = np.max(
        np.abs(gram - np.eye(k))
    )

    if orthogonality_error > 1e-8:
        raise ValueError(
            f"{name}: retained PCA basis is not orthonormal. "
            f"Maximum error = {orthogonality_error:.3e}"
        )

    REFERENCE_SUBSPACES[name] = reference_basis

    print(
        f"{name:28s} "
        f"basis={reference_basis.shape} | "
        f"k={k} | "
        f"orthogonality error={orthogonality_error:.3e}"
    )


# ------------------------------------------------------------
# Single recording-level bootstrap
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE-BOOTSTRAP PRINCIPAL-ANGLE TEST")
print("=" * 70)


# One recording-level resample.
#
# The same recording indices are used for every representation.
# This keeps the bootstrap perturbation identical across
# representations.
bootstrap_indices = rng.integers(
    low=0,
    high=N_RECORDINGS,
    size=N_RECORDINGS
)


single_bootstrap_results = {}


for name in REPRESENTATIONS:

    X = X_REPRESENTATIONS[name]
    k = REFERENCE_PCA[name]["retained_pcs"]

    X_boot = X[bootstrap_indices, :]

    # Refit PCA to the bootstrap recording sample.
    #
    # The 06B representation itself is NOT reprocessed here.
    # This isolates sampling stability of the PCA structure.
    pca_boot = PCA(
        n_components=k,
        svd_solver="full"
    )

    pca_boot.fit(X_boot)

    # Convert sklearn row-oriented components into
    # column-oriented subspace basis vectors.
    bootstrap_basis = pca_boot.components_.T

    # --------------------------------------------------------
    # Principal angles
    # --------------------------------------------------------

    angles_rad = subspace_angles(
        REFERENCE_SUBSPACES[name],
        bootstrap_basis
    )

    angles_deg = np.degrees(angles_rad)

    single_bootstrap_results[name] = angles_deg

    print(
        f"{name:28s} "
        f"min={angles_deg.min():8.3f}° | "
        f"median={np.median(angles_deg):8.3f}° | "
        f"max={angles_deg.max():8.3f}°"
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("06E.3 VALIDATION")
print("=" * 70)

for name, angles in single_bootstrap_results.items():

    k = RETAINED_PCS[name]

    if len(angles) != k:
        raise ValueError(
            f"{name}: expected {k} principal angles, "
            f"found {len(angles)}"
        )

    if not np.isfinite(angles).all():
        raise ValueError(
            f"{name}: principal angles contain non-finite values"
        )

print("✓ All six representations produced valid principal angles")
print("✓ Bootstrap was performed at the recording level")
print("✓ PCA was refit within the bootstrap sample")
print("✓ Reference and bootstrap subspaces have matching dimensions")
print("✓ Principal-angle calculation returned finite values")

LOADING 06B REPRESENTATION MATRICES
baseline                     shape=(275, 50) | features=50
distribution_aware           shape=(275, 50) | features=50
yeo_johnson                  shape=(275, 50) | features=50
block_balanced               shape=(275, 50) | features=50
ndsi_derived_excluded        shape=(275, 48) | features=48
ndsi_family_excluded         shape=(275, 46) | features=46

CONSTRUCTING REFERENCE PCA SUBSPACES
baseline                     basis=(50, 9) | k=9 | orthogonality error=1.554e-15
distribution_aware           basis=(50, 9) | k=9 | orthogonality error=2.331e-15
yeo_johnson                  basis=(50, 9) | k=9 | orthogonality error=1.998e-15
block_balanced               basis=(50, 7) | k=7 | orthogonality error=2.220e-15
ndsi_derived_excluded        basis=(48, 9) | k=9 | orthogonality error=2.665e-15
ndsi_family_excluded         basis=(46, 9) | k=9 | orthogonality error=1.998e-15

SINGLE-BOOTSTRAP PRINCIPAL-ANGLE TEST
baseline                     min=   3.019° | me

In [10]:
# ============================================================
# 06E.4 — Full recording-level bootstrap stability analysis
# ============================================================

import time


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_BOOTSTRAP = 500

BOOTSTRAP_ANGLES = {}
BOOTSTRAP_EV_RATIO = {}

for name in REPRESENTATIONS:

    k = REFERENCE_PCA[name]["retained_pcs"]

    BOOTSTRAP_ANGLES[name] = np.full(
        (N_BOOTSTRAP, k),
        np.nan,
        dtype=float
    )

    BOOTSTRAP_EV_RATIO[name] = np.full(
        (N_BOOTSTRAP, k),
        np.nan,
        dtype=float
    )


# ------------------------------------------------------------
# Independent RNG for the full bootstrap
# ------------------------------------------------------------

bootstrap_rng = np.random.default_rng(42)


print("=" * 70)
print("FULL PCA BOOTSTRAP STABILITY ANALYSIS")
print("=" * 70)

print(f"Representations:       {len(REPRESENTATIONS)}")
print(f"Recordings:             {N_RECORDINGS}")
print(f"Bootstrap replicates:   {N_BOOTSTRAP}")
print("Sampling unit:          recording")
print("Sampling method:        with replacement")
print("Random seed:            42")
print("=" * 70)


start_time = time.time()


# ------------------------------------------------------------
# Bootstrap loop
# ------------------------------------------------------------

for b in range(N_BOOTSTRAP):

    # --------------------------------------------------------
    # One recording-level bootstrap sample
    # --------------------------------------------------------

    bootstrap_indices = bootstrap_rng.integers(
        low=0,
        high=N_RECORDINGS,
        size=N_RECORDINGS
    )


    # --------------------------------------------------------
    # Fit each representation on the same bootstrap sample
    # --------------------------------------------------------

    for name in REPRESENTATIONS:

        X = X_REPRESENTATIONS[name]

        k = REFERENCE_PCA[name]["retained_pcs"]

        X_boot = X[bootstrap_indices, :]


        # ----------------------------------------------------
        # Refit PCA
        # ----------------------------------------------------

        pca_boot = PCA(
            n_components=k,
            svd_solver="full"
        )

        pca_boot.fit(X_boot)


        # ----------------------------------------------------
        # Bootstrap PCA subspace
        # ----------------------------------------------------

        bootstrap_basis = pca_boot.components_.T


        # ----------------------------------------------------
        # Principal angles
        # ----------------------------------------------------

        angles_rad = subspace_angles(
            REFERENCE_SUBSPACES[name],
            bootstrap_basis
        )

        angles_deg = np.degrees(angles_rad)


        if len(angles_deg) != k:
            raise RuntimeError(
                f"{name}: expected {k} principal angles, "
                f"found {len(angles_deg)}"
            )


        BOOTSTRAP_ANGLES[name][b, :] = angles_deg


        # ----------------------------------------------------
        # Explained variance ratio
        # ----------------------------------------------------

        BOOTSTRAP_EV_RATIO[name][b, :] = (
            pca_boot.explained_variance_ratio_
        )


    # --------------------------------------------------------
    # Progress reporting
    # --------------------------------------------------------

    if (b + 1) % 50 == 0:

        elapsed = time.time() - start_time

        print(
            f"Completed {b + 1:3d}/{N_BOOTSTRAP} "
            f"bootstrap replicates "
            f"({elapsed:.1f} s elapsed)"
        )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BOOTSTRAP COMPUTATION COMPLETE")
print("=" * 70)

for name in REPRESENTATIONS:

    angles = BOOTSTRAP_ANGLES[name]
    ev_ratio = BOOTSTRAP_EV_RATIO[name]

    if not np.isfinite(angles).all():
        raise ValueError(
            f"{name}: bootstrap angles contain non-finite values"
        )

    if not np.isfinite(ev_ratio).all():
        raise ValueError(
            f"{name}: bootstrap explained variance contains "
            "non-finite values"
        )

    expected_k = RETAINED_PCS[name]

    if angles.shape != (N_BOOTSTRAP, expected_k):
        raise ValueError(
            f"{name}: unexpected angle shape "
            f"{angles.shape}"
        )

    if ev_ratio.shape != (N_BOOTSTRAP, expected_k):
        raise ValueError(
            f"{name}: unexpected EV-ratio shape "
            f"{ev_ratio.shape}"
        )

    print(
        f"{name:28s} "
        f"angles={angles.shape} | "
        f"EV ratios={ev_ratio.shape}"
    )


elapsed_total = time.time() - start_time

print("\n" + "=" * 70)
print("06E.4 VALIDATION PASSED")
print("=" * 70)

print(
    f"Total computation time: "
    f"{elapsed_total / 60:.2f} minutes"
)

FULL PCA BOOTSTRAP STABILITY ANALYSIS
Representations:       6
Recordings:             275
Bootstrap replicates:   500
Sampling unit:          recording
Sampling method:        with replacement
Random seed:            42
Completed  50/500 bootstrap replicates (1.7 s elapsed)
Completed 100/500 bootstrap replicates (3.2 s elapsed)
Completed 150/500 bootstrap replicates (4.6 s elapsed)
Completed 200/500 bootstrap replicates (5.9 s elapsed)
Completed 250/500 bootstrap replicates (7.1 s elapsed)
Completed 300/500 bootstrap replicates (8.7 s elapsed)
Completed 350/500 bootstrap replicates (10.1 s elapsed)
Completed 400/500 bootstrap replicates (11.3 s elapsed)
Completed 450/500 bootstrap replicates (12.6 s elapsed)
Completed 500/500 bootstrap replicates (14.2 s elapsed)

BOOTSTRAP COMPUTATION COMPLETE
baseline                     angles=(500, 9) | EV ratios=(500, 9)
distribution_aware           angles=(500, 9) | EV ratios=(500, 9)
yeo_johnson                  angles=(500, 9) | EV ratios=(500

In [11]:
# ============================================================
# 06E.5 — Bootstrap principal-angle and variance summary
# ============================================================

print("=" * 70)
print("BOOTSTRAP STABILITY SUMMARY")
print("=" * 70)


# ------------------------------------------------------------
# Helper for percentile summaries
# ------------------------------------------------------------

def summarize_distribution(values):
    """
    Return robust distributional summaries for bootstrap results.
    """

    values = np.asarray(values, dtype=float)

    return {
        "median": np.median(values),
        "q02_5": np.percentile(values, 2.5),
        "q25": np.percentile(values, 25),
        "q75": np.percentile(values, 75),
        "q97_5": np.percentile(values, 97.5),
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "max": np.max(values),
    }


# ------------------------------------------------------------
# Principal-angle summary
# ------------------------------------------------------------

angle_summary_rows = []

for name in REPRESENTATIONS:

    angles = BOOTSTRAP_ANGLES[name]
    k = RETAINED_PCS[name]

    for pc_idx in range(k):

        summary = summarize_distribution(
            angles[:, pc_idx]
        )

        angle_summary_rows.append({
            "representation": name,
            "PC": pc_idx + 1,
            **summary
        })


angle_summary = pd.DataFrame(
    angle_summary_rows
)


# ------------------------------------------------------------
# Explained-variance-ratio summary
# ------------------------------------------------------------

ev_summary_rows = []

for name in REPRESENTATIONS:

    ev_ratio = BOOTSTRAP_EV_RATIO[name]
    k = RETAINED_PCS[name]

    for pc_idx in range(k):

        summary = summarize_distribution(
            ev_ratio[:, pc_idx]
        )

        ev_summary_rows.append({
            "representation": name,
            "PC": pc_idx + 1,
            **summary
        })


ev_ratio_summary = pd.DataFrame(
    ev_summary_rows
)


# ------------------------------------------------------------
# Display principal-angle summaries
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRINCIPAL-ANGLE BOOTSTRAP SUMMARY")
print("=" * 70)

for name in REPRESENTATIONS:

    subset = angle_summary[
        angle_summary["representation"] == name
    ]

    print(f"\n{name.upper()}")

    display(
        subset[
            [
                "PC",
                "median",
                "q02_5",
                "q25",
                "q75",
                "q97_5",
                "max"
            ]
        ].round(3)
    )


# ------------------------------------------------------------
# Display explained-variance summaries
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPLAINED-VARIANCE-RATIO BOOTSTRAP SUMMARY")
print("=" * 70)

for name in REPRESENTATIONS:

    subset = ev_ratio_summary[
        ev_ratio_summary["representation"] == name
    ]

    print(f"\n{name.upper()}")

    display(
        subset[
            [
                "PC",
                "median",
                "q02_5",
                "q25",
                "q75",
                "q97_5"
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_angle_rows = sum(
    RETAINED_PCS.values()
)

if len(angle_summary) != expected_angle_rows:
    raise ValueError(
        "Unexpected number of principal-angle summary rows."
    )

if len(ev_ratio_summary) != expected_angle_rows:
    raise ValueError(
        "Unexpected number of explained-variance summary rows."
    )

if not np.isfinite(
    angle_summary.select_dtypes(include=np.number)
).all().all():
    raise ValueError(
        "Non-finite values found in angle summary."
    )

if not np.isfinite(
    ev_ratio_summary.select_dtypes(include=np.number)
).all().all():
    raise ValueError(
        "Non-finite values found in explained-variance summary."
    )


print("\n" + "=" * 70)
print("06E.5 VALIDATION PASSED")
print("=" * 70)

print(
    f"Principal-angle summaries: "
    f"{len(angle_summary)} rows"
)

print(
    f"Explained-variance summaries: "
    f"{len(ev_ratio_summary)} rows"
)

BOOTSTRAP STABILITY SUMMARY

PRINCIPAL-ANGLE BOOTSTRAP SUMMARY

BASELINE


,PC,median,q02_5,q25,q75,q97_5,max
0,1,61.099,29.678,45.503,74.028,88.232,89.922
1,2,28.438,18.661,23.996,34.778,52.714,70.658
2,3,19.069,13.842,16.927,21.901,30.347,36.177
3,4,14.650,10.716,13.179,16.220,20.589,26.484
4,5,11.227,8.211,10.021,12.482,15.413,19.588
5,6,8.774,6.443,7.919,9.673,11.527,13.618
6,7,6.767,5.020,6.109,7.455,8.864,10.661
7,8,4.845,3.523,4.359,5.491,6.509,7.305
8,9,3.274,2.277,2.880,3.718,4.652,5.344



DISTRIBUTION_AWARE


,PC,median,q02_5,q25,q75,q97_5,max
9,1,53.097,25.872,37.981,71.754,88.324,89.681
10,2,24.954,16.259,21.090,30.049,43.012,69.822
11,3,16.854,12.413,14.966,18.974,24.566,28.499
12,4,12.382,9.297,11.019,13.643,16.646,18.741
13,5,9.505,7.156,8.644,10.389,12.892,15.885
14,6,7.390,5.797,6.683,8.051,9.463,11.690
15,7,5.706,4.228,5.243,6.288,7.354,9.015
16,8,4.165,3.207,3.798,4.627,5.496,5.915
17,9,2.941,2.154,2.616,3.279,4.153,4.506



YEO_JOHNSON


...


NDSI_DERIVED_EXCLUDED


,PC,median,q02_5,q25,q75,q97_5
34,1,0.2216,0.1910,0.2108,0.2330,0.2596
35,2,0.1248,0.1091,0.1181,0.1307,0.1412
36,3,0.0785,0.0681,0.0744,0.0819,0.0895
37,4,0.0661,0.0591,0.0635,0.0693,0.0752
38,5,0.0585,0.0513,0.0556,0.0609,0.0653
39,6,0.0504,0.0441,0.0480,0.0531,0.0581
40,7,0.0432,0.0382,0.0414,0.0453,0.0495
41,8,0.0376,0.0331,0.0360,0.0394,0.0428
42,9,0.0330,0.0289,0.0316,0.0346,0.0378



NDSI_FAMILY_EXCLUDED


,PC,median,q02_5,q25,q75,q97_5
43,1,0.2290,0.1974,0.2182,0.2408,0.2678
44,2,0.1245,0.1089,0.1188,0.1309,0.1427
45,3,0.0773,0.0675,0.0734,0.0811,0.0891
46,4,0.0660,0.0586,0.0634,0.0694,0.0751
47,5,0.0570,0.0496,0.0544,0.0600,0.0652
48,6,0.0489,0.0432,0.0468,0.0515,0.0573
49,7,0.0427,0.0379,0.0410,0.0446,0.0489
50,8,0.0377,0.0334,0.0359,0.0392,0.0422
51,9,0.0333,0.0294,0.0319,0.0349,0.0377



06E.5 VALIDATION PASSED
Principal-angle summaries: 52 rows
Explained-variance summaries: 52 rows


In [12]:
# ============================================================
# 06E.6 — Subspace-level bootstrap stability summaries
# ============================================================

print("=" * 70)
print("SUBSPACE-LEVEL BOOTSTRAP STABILITY")
print("=" * 70)


# ------------------------------------------------------------
# Principal-angle geometry
# ------------------------------------------------------------
#
# For a k-dimensional reference subspace and bootstrap
# subspace, scipy returns k principal angles:
#
#   theta_1, ..., theta_k
#
# These are NOT individual PCA-axis uncertainty estimates.
# They describe the relative geometry of the two subspaces.
#
# We therefore summarize the whole retained subspace rather
# than labeling individual angles as PC1, PC2, etc.


subspace_summary_rows = []


for name in REPRESENTATIONS:

    angles_deg = BOOTSTRAP_ANGLES[name]

    # --------------------------------------------------------
    # Convert to radians for cosine-based similarity
    # --------------------------------------------------------

    angles_rad = np.radians(angles_deg)

    # --------------------------------------------------------
    # Subspace-level summaries for each bootstrap replicate
    # --------------------------------------------------------

    max_angle = np.max(
        angles_deg,
        axis=1
    )

    median_angle = np.median(
        angles_deg,
        axis=1
    )

    mean_angle = np.mean(
        angles_deg,
        axis=1
    )

    rms_angle = np.sqrt(
        np.mean(
            angles_deg ** 2,
            axis=1
        )
    )

    # Average squared cosine similarity across principal
    # angles. Values closer to 1 indicate greater alignment.
    cosine_similarity = np.mean(
        np.cos(angles_rad) ** 2,
        axis=1
    )


    # --------------------------------------------------------
    # Summarize each bootstrap distribution
    # --------------------------------------------------------

    metrics = {
        "max_angle_deg": max_angle,
        "median_angle_deg": median_angle,
        "mean_angle_deg": mean_angle,
        "rms_angle_deg": rms_angle,
        "mean_squared_cosine": cosine_similarity,
    }


    for metric_name, values in metrics.items():

        subspace_summary_rows.append({
            "representation": name,
            "metric": metric_name,
            "median": np.median(values),
            "q02_5": np.percentile(values, 2.5),
            "q25": np.percentile(values, 25),
            "q75": np.percentile(values, 75),
            "q97_5": np.percentile(values, 97.5),
            "mean": np.mean(values),
            "std": np.std(values, ddof=1),
        })


subspace_summary = pd.DataFrame(
    subspace_summary_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

for name in REPRESENTATIONS:

    subset = subspace_summary[
        subspace_summary["representation"] == name
    ]

    print("\n" + "-" * 70)
    print(name.upper())
    print("-" * 70)

    display(
        subset[
            [
                "metric",
                "median",
                "q02_5",
                "q25",
                "q75",
                "q97_5",
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_rows = (
    len(REPRESENTATIONS) * 5
)

if len(subspace_summary) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} summary rows, "
        f"found {len(subspace_summary)}"
    )


numeric_columns = [
    "median",
    "q02_5",
    "q25",
    "q75",
    "q97_5",
    "mean",
    "std",
]

if not np.isfinite(
    subspace_summary[numeric_columns].to_numpy()
).all():
    raise ValueError(
        "Subspace summary contains non-finite values."
    )


# Cosine similarity must lie in [0, 1].
cosine_rows = subspace_summary[
    subspace_summary["metric"] == "mean_squared_cosine"
]

if not (
    (cosine_rows["median"] >= 0).all()
    and
    (cosine_rows["median"] <= 1).all()
):
    raise ValueError(
        "Invalid cosine similarity values."
    )


print("\n" + "=" * 70)
print("06E.6 VALIDATION PASSED")
print("=" * 70)

print(
    "✓ Subspace geometry summarized without treating "
    "individual principal angles as PCA-axis uncertainty"
)

print(
    "✓ Maximum, median, mean, and RMS angles calculated"
)

print(
    "✓ Mean squared-cosine subspace similarity calculated"
)

print(
    f"✓ {len(subspace_summary)} summary rows generated"
)

SUBSPACE-LEVEL BOOTSTRAP STABILITY

----------------------------------------------------------------------
BASELINE
----------------------------------------------------------------------


,metric,median,q02_5,q25,q75,q97_5
0,max_angle_deg,61.0991,29.6777,45.5033,74.0278,88.2316
1,median_angle_deg,11.2270,8.2113,10.0214,12.4819,15.4128
2,mean_angle_deg,17.8479,12.4115,15.3032,19.8844,23.8465
3,rms_angle_deg,24.7418,15.2119,19.9644,29.1869,34.7042
4,mean_squared_cosine,0.8575,0.7805,0.8337,0.8957,0.9337



----------------------------------------------------------------------
DISTRIBUTION_AWARE
----------------------------------------------------------------------


,metric,median,q02_5,q25,q75,q97_5
5,max_angle_deg,53.0969,25.8717,37.9814,71.7544,88.3236
6,median_angle_deg,9.5047,7.1565,8.6442,10.3892,12.8917
7,mean_angle_deg,15.2570,10.9106,13.4323,18.0115,21.1431
8,rms_angle_deg,21.7320,13.1710,17.3340,27.3548,32.9018
9,mean_squared_cosine,0.8852,0.8150,0.8538,0.9173,0.9492



----------------------------------------------------------------------
YEO_JOHNSON
----------------------------------------------------------------------


...


----------------------------------------------------------------------
NDSI_DERIVED_EXCLUDED
----------------------------------------------------------------------


,metric,median,q02_5,q25,q75,q97_5
20,max_angle_deg,56.3907,29.4683,42.0666,71.2065,88.2217
21,median_angle_deg,10.9644,8.0640,9.7351,12.1574,14.7523
22,mean_angle_deg,17.1809,12.1473,14.9397,19.5085,23.0207
23,rms_angle_deg,23.3216,14.9586,19.2359,28.6580,33.9968
24,mean_squared_cosine,0.8687,0.7903,0.8377,0.9011,0.9356



----------------------------------------------------------------------
NDSI_FAMILY_EXCLUDED
----------------------------------------------------------------------


,metric,median,q02_5,q25,q75,q97_5
25,max_angle_deg,59.7758,31.9883,44.8023,72.9795,88.3807
26,median_angle_deg,10.7797,7.9980,9.6979,11.9845,15.0336
27,mean_angle_deg,17.5439,12.5142,15.2908,19.7724,23.6232
28,rms_angle_deg,24.7407,15.5097,20.0083,29.0825,34.3577
29,mean_squared_cosine,0.8593,0.7857,0.8336,0.8942,0.9310



06E.6 VALIDATION PASSED
✓ Subspace geometry summarized without treating individual principal angles as PCA-axis uncertainty
✓ Maximum, median, mean, and RMS angles calculated
✓ Mean squared-cosine subspace similarity calculated
✓ 30 summary rows generated


In [13]:
# ============================================================
# 06E.7 — Compact cross-representation stability comparison
# ============================================================

comparison_metrics = [
    "max_angle_deg",
    "median_angle_deg",
    "mean_angle_deg",
    "rms_angle_deg",
    "mean_squared_cosine",
]

comparison = (
    subspace_summary[
        subspace_summary["metric"].isin(comparison_metrics)
    ]
    .pivot(
        index="representation",
        columns="metric",
        values="median"
    )
    .reset_index()
)

# Reorder columns for readability
comparison = comparison[
    [
        "representation",
        "max_angle_deg",
        "median_angle_deg",
        "mean_angle_deg",
        "rms_angle_deg",
        "mean_squared_cosine",
    ]
]

display(comparison.round(4))

print("\n" + "=" * 70)
print("06E.7 VALIDATION PASSED")
print("=" * 70)

print(
    "✓ Compact six-representation comparison generated "
    "from existing bootstrap results"
)

metric,representation,max_angle_deg,median_angle_deg,mean_angle_deg,rms_angle_deg,mean_squared_cosine
0,baseline,61.0991,11.2270,17.8479,24.7418,0.8575
1,block_balanced,35.3056,9.0983,13.5009,17.0766,0.9190
2,distribution_aware,53.0969,9.5047,15.2570,21.7320,0.8852
3,ndsi_derived_excluded,56.3907,10.9644,17.1809,23.3216,0.8687
4,ndsi_family_excluded,59.7758,10.7797,17.5439,24.7407,0.8593
5,yeo_johnson,45.3162,8.6848,13.6897,18.8487,0.9078



06E.7 VALIDATION PASSED
✓ Compact six-representation comparison generated from existing bootstrap results


In [14]:
# ============================================================
# 06E.8 — Save 06E bootstrap stability artifacts
# ============================================================

print("=" * 70)
print("SAVING 06E BOOTSTRAP ARTIFACTS")
print("=" * 70)


# ------------------------------------------------------------
# Save bootstrap arrays
# ------------------------------------------------------------

for name in REPRESENTATIONS:

    np.save(
        OUTPUT_DIR / f"{name}_principal_angles_deg.npy",
        BOOTSTRAP_ANGLES[name]
    )

    np.save(
        OUTPUT_DIR / f"{name}_bootstrap_explained_variance_ratio.npy",
        BOOTSTRAP_EV_RATIO[name]
    )


# ------------------------------------------------------------
# Save summary tables
# ------------------------------------------------------------

angle_summary_path = (
    OUTPUT_DIR / "principal_angle_summary.csv"
)

ev_summary_path = (
    OUTPUT_DIR / "bootstrap_explained_variance_summary.csv"
)

subspace_summary_path = (
    OUTPUT_DIR / "subspace_stability_summary.csv"
)

comparison_path = (
    OUTPUT_DIR / "subspace_stability_comparison.csv"
)


angle_summary.to_csv(
    angle_summary_path,
    index=False
)

ev_ratio_summary.to_csv(
    ev_summary_path,
    index=False
)

subspace_summary.to_csv(
    subspace_summary_path,
    index=False
)

comparison.to_csv(
    comparison_path,
    index=False
)


# ------------------------------------------------------------
# Save analysis configuration
# ------------------------------------------------------------

config = {
    "analysis": "PCA bootstrap stability",
    "n_recordings": N_RECORDINGS,
    "n_bootstrap": N_BOOTSTRAP,
    "random_seed": 42,
    "sampling_unit": "recording",
    "sampling_method": "bootstrap_with_replacement",
    "representations": REPRESENTATIONS,
    "retained_pcs": RETAINED_PCS,
    "preprocessing_refit_within_bootstrap": False,
    "principal_angle_method": (
        "scipy.linalg.subspace_angles"
    ),
    "angle_units": "degrees",
    "subspace_similarity": (
        "mean_squared_cosine_of_principal_angles"
    ),
}


config_path = (
    OUTPUT_DIR / "bootstrap_stability_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

saved_files = sorted(
    OUTPUT_DIR.iterdir()
)

print("\nSaved files:")

for path in saved_files:
    print(f"✓ {path.name}")


print("\n" + "=" * 70)
print("06E.8 SAVE VALIDATION PASSED")
print("=" * 70)

print(
    f"Files saved: {len(saved_files)}"
)

print(
    f"Output directory:\n{OUTPUT_DIR}"
)

SAVING 06E BOOTSTRAP ARTIFACTS

Saved files:
✓ baseline_bootstrap_explained_variance_ratio.npy
✓ baseline_principal_angles_deg.npy
✓ block_balanced_bootstrap_explained_variance_ratio.npy
✓ block_balanced_principal_angles_deg.npy
✓ bootstrap_explained_variance_summary.csv
✓ bootstrap_stability_config.json
✓ distribution_aware_bootstrap_explained_variance_ratio.npy
✓ distribution_aware_principal_angles_deg.npy
✓ ndsi_derived_excluded_bootstrap_explained_variance_ratio.npy
✓ ndsi_derived_excluded_principal_angles_deg.npy
✓ ndsi_family_excluded_bootstrap_explained_variance_ratio.npy
✓ ndsi_family_excluded_principal_angles_deg.npy
✓ principal_angle_summary.csv
✓ subspace_stability_comparison.csv
✓ subspace_stability_summary.csv
✓ yeo_johnson_bootstrap_explained_variance_ratio.npy
✓ yeo_johnson_principal_angles_deg.npy

06E.8 SAVE VALIDATION PASSED
Files saved: 17
Output directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\bootstrap_stability
